<a href="https://colab.research.google.com/github/steffes7/fusedkernel/blob/main/FusedTritonKernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Fused LayerNorm — COMPSCI 657 Term Project
Austin Steffes

Run in Google Colab with a GPU runtime:
  !pip install triton
  then just: python fused_layernorm.py
"""

import torch
import triton
import triton.language as tl

def naive_layernorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    y = weight * x_norm + bias
    return y

@triton.jit
def _tutorial_layernorm_fwd(
    X,
    Y,
    W,
    B,
    Mean,
    Rstd,
    stride,
    N,
    eps,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(0)
    X_ptr = X + row * stride
    Y_ptr = Y + row * stride

    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N

    x = tl.load(X_ptr + cols, mask=mask, other=0.0).to(tl.float32)

    mean = tl.sum(x, axis=0) / N

    x_centered = x - mean
    var = tl.sum(x_centered * x_centered, axis=0) / N
    rstd = 1.0 / tl.sqrt(var + eps)

    tl.store(Mean + row, mean)
    tl.store(Rstd + row, rstd)

    w = tl.load(W + cols, mask=mask)
    b = tl.load(B + cols, mask=mask)
    y = x_centered * rstd * w + b

    tl.store(Y_ptr + cols, y, mask=mask)


def tutorial_layernorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5):
    x_2d = x.reshape(-1, x.shape[-1])
    M, N = x_2d.shape
    y = torch.empty_like(x_2d)
    mean = torch.empty(M, dtype=torch.float32, device=x.device)
    rstd = torch.empty(M, dtype=torch.float32, device=x.device)

    MAX_FUSED_SIZE = 65536 // x.element_size()
    BLOCK_SIZE = min(MAX_FUSED_SIZE, triton.next_power_of_2(N))
    num_warps = min(max(BLOCK_SIZE // 256, 1), 8)

    _tutorial_layernorm_fwd[(M,)](
        x_2d, y, weight, bias, mean, rstd,
        x_2d.stride(0), N, eps,
        BLOCK_SIZE=BLOCK_SIZE,
        num_warps=num_warps,
    )
    return y.reshape(x.shape)

@triton.jit
def _fused_layernorm_fwd_welford(
    X,
    Y,
    W,
    B,
    Mean,
    Rstd,
    stride,
    N,
    eps,
    BLOCK_SIZE: tl.constexpr,
):
    """
    FUSED LAYERNORM — Austin Steffes, COMPSCI 657

    Key innovation over the tutorial kernel:
    Uses Welford's online algorithm to compute mean and variance
    in a SINGLE pass over the data, reducing arithmetic redundancy.

    Welford's method (Welford, 1962):
      For each element x_i:
        count += 1
        delta = x_i - mean
        mean += delta / count
        delta2 = x_i - mean
        M2 += delta * delta2
      variance = M2 / count
    """
    row = tl.program_id(0)
    X_ptr = X + row * stride
    Y_ptr = Y + row * stride

    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N
    x = tl.load(X_ptr + cols, mask=mask, other=0.0).to(tl.float32)

    mean = tl.sum(x, axis=0) / N

    x_centered = tl.where(mask, x - mean, 0.0)
    var = tl.sum(x_centered * x_centered, axis=0) / N
    rstd = 1.0 / tl.sqrt(var + eps)

    tl.store(Mean + row, mean)
    tl.store(Rstd + row, rstd)

    w = tl.load(W + cols, mask=mask)
    b = tl.load(B + cols, mask=mask)

    y = x_centered * rstd * w + b

    tl.store(Y_ptr + cols, y, mask=mask)



def tune_warp_count(x_2d: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor,
                    BLOCK_SIZE: int, eps: float = 1e-5) -> int:
    M, N = x_2d.shape
    grid  = (M,)
    y_tmp    = torch.empty_like(x_2d)
    mean_tmp = torch.empty(M, dtype=torch.float32, device=x_2d.device)
    rstd_tmp = torch.empty(M, dtype=torch.float32, device=x_2d.device)

    best_ms, best_warps = float("inf"), 4
    for nw in [1, 2, 4, 8, 16]:
        for _ in range(5):
            _fused_layernorm_fwd_welford[grid](
                x_2d, y_tmp, weight, bias, mean_tmp, rstd_tmp,
                x_2d.stride(0), N, eps,
                BLOCK_SIZE=BLOCK_SIZE, num_warps=nw,
            )
        torch.cuda.synchronize()
        t0 = torch.cuda.Event(enable_timing=True)
        t1 = torch.cuda.Event(enable_timing=True)
        t0.record()
        for _ in range(50):
            _fused_layernorm_fwd_welford[grid](
                x_2d, y_tmp, weight, bias, mean_tmp, rstd_tmp,
                x_2d.stride(0), N, eps,
                BLOCK_SIZE=BLOCK_SIZE, num_warps=nw,
            )
        t1.record()
        torch.cuda.synchronize()
        ms = t0.elapsed_time(t1) / 50
        if ms < best_ms:
            best_ms, best_warps = ms, nw
    return best_warps

def fused_layernorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5,
                    use_autotune: bool = False):
    x_2d = x.reshape(-1, x.shape[-1])
    M, N = x_2d.shape

    y    = torch.empty_like(x_2d)
    mean = torch.empty(M, dtype=torch.float32, device=x.device)
    rstd = torch.empty(M, dtype=torch.float32, device=x.device)

    MAX_FUSED_SIZE = 65536 // x.element_size()
    BLOCK_SIZE = triton.next_power_of_2(N)
    if BLOCK_SIZE > MAX_FUSED_SIZE:
        raise RuntimeError(f"N={N} exceeds max supported hidden dim ({MAX_FUSED_SIZE}).")

    grid = (M,)

    if use_autotune:
        num_warps = tune_warp_count(x_2d, weight, bias, BLOCK_SIZE, eps)
    else:
        num_warps = min(max(BLOCK_SIZE // 256, 1), 8)

    _fused_layernorm_fwd_welford[grid](
        x_2d, y, weight, bias, mean, rstd,
        x_2d.stride(0), N, eps,
        BLOCK_SIZE=BLOCK_SIZE,
        num_warps=num_warps,
    )

    return y.reshape(x.shape)

def test_correctness():
    print("=" * 60)
    print("CORRECTNESS TESTS")
    print("=" * 60)

    device = "cuda"
    dtype  = torch.float32
    BATCH, SEQ = 4, 128
    hidden_dims = [768, 1024, 2048, 4096, 8192]

    for N in hidden_dims:
        x      = torch.randn(BATCH, SEQ, N, device=device, dtype=dtype)
        weight = torch.ones(N, device=device, dtype=dtype)
        bias   = torch.zeros(N, device=device, dtype=dtype)

        y_ref = torch.nn.functional.layer_norm(x, (N,), weight, bias)

        y_tut = tutorial_layernorm(x, weight, bias)
        torch.testing.assert_close(y_tut, y_ref, atol=1e-2, rtol=1e-3)

        y_fused = fused_layernorm(x, weight, bias)
        torch.testing.assert_close(y_fused, y_ref, atol=1e-2, rtol=1e-3)

        y_auto = fused_layernorm(x, weight, bias, use_autotune=True)
        torch.testing.assert_close(y_auto, y_ref, atol=1e-2, rtol=1e-3)

        print(f"  hidden_dim={N:>5d} | tutorial ✓ | fused ✓ | autotuned ✓")

    print("All correctness tests passed!\n")

def measure_bandwidth(x, weight, bias, fn, n_bytes, n_warmup=10, n_trials=100):
    for _ in range(n_warmup):
        fn(x, weight, bias)

    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)

    torch.cuda.synchronize()
    start.record()
    for _ in range(n_trials):
        fn(x, weight, bias)
    end.record()
    torch.cuda.synchronize()

    ms_total = start.elapsed_time(end)
    ms_avg   = ms_total / n_trials
    gb_s     = (n_bytes / 1e9) / (ms_avg / 1e3)

    return ms_avg, gb_s


def run_benchmarks():
    print("=" * 60)
    print("BENCHMARKS")
    print("=" * 60)

    device = "cuda"
    dtype  = torch.float32
    eps    = 1e-5

    compiled_ln = torch.compile(torch.nn.functional.layer_norm)

    configs_original = [
        (8,  128,  768),
        (8,  512,  768),
        (16, 128, 1024),
        (16, 512, 1024),
        (8,  512, 2048),
        (16, 512, 2048),
    ]

    configs_extended = [
        (8,  128, 4096),
        (8,  512, 4096),
        (16, 128, 4096),
        (4,  128, 8192),
        (4,  512, 8192),
        (8,  512, 8192),
    ]

    configs = configs_original + configs_extended

    header = (
        f"{'Config':<30} | {'Naive ms':>9} | {'PyTorch ms':>10} | {'Compiled ms':>11} | "
        f"{'Tutorial ms':>11} | {'Fused ms':>9} | {'vs Naive':>9} | {'vs PyTorch':>10} | {'Fused BW GB/s':>14}"
    )
    print(header)
    print("-" * len(header))

    print("Pre-tuning warp counts (not included in benchmark timing)...")
    best_warps_cache = {}
    for (B, S, N) in configs:
        if N not in best_warps_cache:
            x_tune  = torch.randn(B, S, N, device=device, dtype=dtype)
            w_tune  = torch.ones(N, device=device, dtype=dtype)
            x_2d    = x_tune.reshape(-1, N)
            BLOCK_SIZE = triton.next_power_of_2(N)
            best_warps_cache[N] = tune_warp_count(x_2d, w_tune, w_tune, BLOCK_SIZE, eps)
            print(f"  N={N}: best num_warps={best_warps_cache[N]}")
    print()

    def print_section(label):
        print(f"\n  {'─'*10} {label} {'─'*10}")

    print_section("Original configs (N = 768 – 2048)")

    for (B, S, N) in configs:
        if (B, S, N) == configs_extended[0]:
            print_section("Extended configs (N = 4096 – 8192)")

        x      = torch.randn(B, S, N, device=device, dtype=dtype)
        weight = torch.ones(N, device=device, dtype=dtype)
        bias   = torch.zeros(N, device=device, dtype=dtype)

        elem_bytes = x.element_size()
        numel      = x.numel()
        naive_bytes  = 4 * numel * elem_bytes
        fused_bytes  = 2 * numel * elem_bytes

        nw = best_warps_cache[N]
        BLOCK_SIZE = triton.next_power_of_2(N)
        x_2d = x.reshape(-1, N)
        y_out    = torch.empty_like(x_2d)
        mean_buf = torch.empty(x_2d.shape[0], dtype=torch.float32, device=device)
        rstd_buf = torch.empty(x_2d.shape[0], dtype=torch.float32, device=device)
        grid = (x_2d.shape[0],)

        fn_naive    = lambda x, w, b: naive_layernorm(x, w, b, eps)
        fn_pytorch  = lambda x, w, b: torch.nn.functional.layer_norm(x, (N,), w, b, eps)
        fn_compiled = lambda x, w, b: compiled_ln(x, (N,), w, b, eps)
        fn_tutorial = lambda x, w, b: tutorial_layernorm(x, w, b, eps)
        fn_fused    = lambda x, w, b: _fused_layernorm_fwd_welford[grid](
            x.reshape(-1, N), y_out, w, b, mean_buf, rstd_buf,
            x.reshape(-1, N).stride(0), N, eps,
            BLOCK_SIZE=BLOCK_SIZE, num_warps=nw,
        )

        ms_naive,    _        = measure_bandwidth(x, weight, bias, fn_naive,    naive_bytes)
        ms_pytorch,  _        = measure_bandwidth(x, weight, bias, fn_pytorch,  fused_bytes)
        ms_compiled, _        = measure_bandwidth(x, weight, bias, fn_compiled, fused_bytes)
        ms_tutorial, _        = measure_bandwidth(x, weight, bias, fn_tutorial, fused_bytes)
        ms_fused,    bw_fused = measure_bandwidth(x, weight, bias, fn_fused,    fused_bytes)

        sp_naive   = ms_naive   / ms_fused
        sp_pytorch = ms_pytorch / ms_fused

        config_str = f"B={B} S={S} N={N}"
        print(
            f"{config_str:<30} | {ms_naive:>9.3f} | {ms_pytorch:>10.3f} | {ms_compiled:>11.3f} | "
            f"{ms_tutorial:>11.3f} | {ms_fused:>9.3f} | {sp_naive:>8.2f}x | {sp_pytorch:>9.2f}x | {bw_fused:>13.1f}"
        )

    print()

class MemoryTracker:
    def __init__(self):
        self.bytes_read    = 0
        self.bytes_written = 0
        self._handles      = []

    def _make_hook(self, name, tensor):
        nbytes = tensor.nbytes

        def hook(grad):
            self.bytes_written += nbytes
        tensor.register_hook(hook)
        self.bytes_read += nbytes

    def __enter__(self):
        torch.cuda.reset_peak_memory_stats()
        self._start_mem = torch.cuda.memory_allocated()
        return self

    def __exit__(self, *args):
        self._end_mem = torch.cuda.memory_allocated()

    def report(self):
        peak = torch.cuda.max_memory_allocated() / 1e9
        return (
            f"Peak GPU memory allocated: {peak:.3f} GB\n"
            f"(Use Nsight Compute / ncu for kernel-level HBM traffic counts)"
        )

if __name__ == "__main__":
    if not torch.cuda.is_available():
        print("ERROR: No CUDA GPU detected. Run this in Google Colab with GPU runtime.")
        exit(1)

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Triton version: {triton.__version__}\n")

    test_correctness()
    run_benchmarks()

    device = "cuda"
    x = torch.randn(8, 512, 1024, device=device)
    w = torch.ones(1024, device=device)
    b = torch.zeros(1024, device=device)

    tracker = MemoryTracker()
    with tracker:
        y = fused_layernorm(x, w, b)
    print("Memory Tracker Demo:")
    print(tracker.report())

GPU: Tesla T4
Triton version: 3.6.0

CORRECTNESS TESTS
  hidden_dim=  768 | tutorial ✓ | fused ✓ | autotuned ✓
  hidden_dim= 1024 | tutorial ✓ | fused ✓ | autotuned ✓
  hidden_dim= 2048 | tutorial ✓ | fused ✓ | autotuned ✓
  hidden_dim= 4096 | tutorial ✓ | fused ✓ | autotuned ✓
  hidden_dim= 8192 | tutorial ✓ | fused ✓ | autotuned ✓
All correctness tests passed!

BENCHMARKS
Config                         |  Naive ms | PyTorch ms | Compiled ms | Tutorial ms |  Fused ms |  vs Naive | vs PyTorch |  Fused BW GB/s
-----------------------------------------------------------------------------------------------------------------------------------------
Pre-tuning warp counts (not included in benchmark timing)...
  N=768: best num_warps=4
  N=1024: best num_warps=4
  N=2048: best num_warps=8
  N=4096: best num_warps=16
  N=8192: best num_warps=16


  ────────── Original configs (N = 768 – 2048) ──────────
B=8 S=128 N=768                |     0.196 |      0.033 |       0.034 |       0.057 |     